# Taller práctico de análisis de datos con Pandas
## Ventas en librerías y tiendas culturales — Letras y Café

**Objetivo:** aplicar lectura de CSV, selección, consultas, limpieza, imputación, transformación y agrupación con Pandas.

> Ejecuta las celdas en orden. El archivo `ventas_libreria_cultural.csv` debe estar en la misma carpeta que este notebook.


## 1. Importación y verificación de la base de datos

In [1]:
import pandas as pd
from pathlib import Path

ruta_csv = Path("ventas_libreria_cultural.csv")
if not ruta_csv.exists():
    raise FileNotFoundError(
        "No se encontró ventas_libreria_cultural.csv. "
        "Guarda el CSV en la misma carpeta que este notebook."
    )

# index_col=False evita interpretar una columna del archivo como índice.
df_libros = pd.read_csv(ruta_csv, index_col=False)

print("Dimensiones (filas, columnas):", df_libros.shape)
print("\\nPrimeras cinco filas:")
display(df_libros.head())
print("\\nTipos de datos y valores no nulos:")
df_libros.info()
print("\\nValores ausentes por columna:")
display(df_libros.isna().sum().to_frame("Valores ausentes"))


Dimensiones (filas, columnas): (1012, 8)
\nPrimeras cinco filas:


,ID_Transaccion,Sucursal,Genero_Libro,Formato,Precio_Lista,Unidades_Vendidas,Descuento_Aplicado,Metodo_Pago
0,LIB-9001,Galería de Arte,Ensayo Científico,Bolsillo,72000.0,1,0.00,Efectivo
1,LIB-9002,Centro Histórico,Historia y Crítica,Edición Ilustrada,45000.0,1,0.10,Efectivo
2,LIB-9003,Terminal,Novela Gráfica,Bolsillo,115000.0,1,0.00,Tarjeta Débito
3,LIB-9004,Galería de Arte,Ensayo Científico,Tapa Blanda,45000.0,5,0.00,Tarjeta Crédito
4,LIB-9005,Centro Histórico,Literatura Clásica,Tapa Blanda,32000.0,2,0.05,Tarjeta Crédito


\nTipos de datos y valores no nulos:
<class 'pandas.DataFrame'>
RangeIndex: 1012 entries, 0 to 1011
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID_Transaccion      1012 non-null   str    
 1   Sucursal            1012 non-null   str    
 2   Genero_Libro        1012 non-null   str    
 3   Formato             1012 non-null   str    
 4   Precio_Lista        962 non-null    float64
 5   Unidades_Vendidas   1012 non-null   int64  
 6   Descuento_Aplicado  923 non-null    float64
 7   Metodo_Pago         1012 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 63.4 KB
\nValores ausentes por columna:


,Valores ausentes
ID_Transaccion,0
Sucursal,0
Genero_Libro,0
Formato,0
Precio_Lista,50
Unidades_Vendidas,0
Descuento_Aplicado,89
Metodo_Pago,0


## 2. Selección unidimensional y multivariable
Se extrae `Genero_Libro` como una Serie y se crea otra tabla con `Sucursal`, `Formato` y `Precio_Lista`.


In [2]:
serie_generos = df_libros["Genero_Libro"]
tabla_sucursal_formato_precio = df_libros[["Sucursal", "Formato", "Precio_Lista"]]

print("Tipo de serie:", type(serie_generos))
display(serie_generos.head())
display(tabla_sucursal_formato_precio.head())


Tipo de serie: <class 'pandas.Series'>


0     Ensayo Científico
1    Historia y Crítica
2        Novela Gráfica
3     Ensayo Científico
4    Literatura Clásica
Name: Genero_Libro, dtype: str

,Sucursal,Formato,Precio_Lista
0,Galería de Arte,Bolsillo,72000.0
1,Centro Histórico,Edición Ilustrada,45000.0
2,Terminal,Bolsillo,115000.0
3,Galería de Arte,Tapa Blanda,45000.0
4,Centro Histórico,Tapa Blanda,32000.0


## 3. Acceso por etiqueta con `.loc[]`
Se consulta la fila que tiene exactamente la etiqueta de índice `310`. En el CSV leído sin asignar una columna índice, las etiquetas iniciales corresponden a las filas numeradas desde 0.


In [3]:
fila_310 = df_libros.loc[310]
display(fila_310.to_frame(name="Valor"))


,Valor
ID_Transaccion,LIB-9311
Sucursal,Campus Universitario
Genero_Libro,Historia y Crítica
Formato,Tapa Dura
Precio_Lista,58000.0
Unidades_Vendidas,1
Descuento_Aplicado,0.1
Metodo_Pago,Tarjeta Débito


## 4. Tratamiento de redundancias
Se eliminan filas duplicadas completas con `.drop_duplicates()` y se compara el tamaño antes y después.


In [4]:
filas_antes = len(df_libros)
df_libros = df_libros.drop_duplicates()
filas_despues = len(df_libros)

print("Filas antes de eliminar duplicados:", filas_antes)
print("Filas después de eliminar duplicados:", filas_despues)
print("Filas duplicadas eliminadas:", filas_antes - filas_despues)


Filas antes de eliminar duplicados: 1012
Filas después de eliminar duplicados: 1000
Filas duplicadas eliminadas: 12


## 5. Filtrado por volumen de compra
Se seleccionan compras mayoristas con más de 3 unidades.


In [5]:
compras_mayoristas = df_libros[df_libros["Unidades_Vendidas"] > 3]
print("Cantidad de compras con más de 3 unidades:", len(compras_mayoristas))
display(compras_mayoristas.head())


Cantidad de compras con más de 3 unidades: 476


,ID_Transaccion,Sucursal,Genero_Libro,Formato,Precio_Lista,Unidades_Vendidas,Descuento_Aplicado,Metodo_Pago
3,LIB-9004,Galería de Arte,Ensayo Científico,Tapa Blanda,45000.0,5,0.0,Tarjeta Crédito
5,LIB-9006,Zona Rosa,Literatura Clásica,Edición Ilustrada,89000.0,4,0.0,Transferencia QR
8,LIB-9009,Terminal,Ciencia Ficción,Edición Ilustrada,58000.0,5,0.0,Tarjeta Crédito
9,LIB-9010,Zona Rosa,Historia y Crítica,Edición Ilustrada,32000.0,5,0.0,Transferencia QR
10,LIB-9011,Terminal,Ensayo Científico,Bolsillo,58000.0,4,0.0,Tarjeta Débito


## 6. Filtrado categórico con `.isin()`
Se consultan las operaciones de las sucursales Campus Universitario y Galería de Arte.


In [6]:
sucursales_seleccionadas = ["Campus Universitario", "Galería de Arte"]
ventas_sucursales = df_libros[df_libros["Sucursal"].isin(sucursales_seleccionadas)]
print("Operaciones encontradas:", len(ventas_sucursales))
display(ventas_sucursales.head())


Operaciones encontradas: 428


,ID_Transaccion,Sucursal,Genero_Libro,Formato,Precio_Lista,Unidades_Vendidas,Descuento_Aplicado,Metodo_Pago
0,LIB-9001,Galería de Arte,Ensayo Científico,Bolsillo,72000.0,1,0.0,Efectivo
3,LIB-9004,Galería de Arte,Ensayo Científico,Tapa Blanda,45000.0,5,0.0,Tarjeta Crédito
12,LIB-9013,Campus Universitario,Novela Gráfica,Bolsillo,72000.0,5,NaN,Tarjeta Débito
13,LIB-9014,Campus Universitario,Novela Gráfica,Tapa Dura,45000.0,1,NaN,Tarjeta Crédito
15,LIB-9016,Galería de Arte,Historia y Crítica,Bolsillo,45000.0,4,0.0,Tarjeta Débito


## 7. Filtrado por rango de precios con `.between()`
Se incluyen los límites de $45.000 y $75.000. Los precios ausentes no cumplen esta condición.


In [7]:
ventas_rango_precios = df_libros[
    df_libros["Precio_Lista"].between(45000, 75000, inclusive="both")
]
print("Operaciones entre $45.000 y $75.000:", len(ventas_rango_precios))
display(ventas_rango_precios.head())


Operaciones entre $45.000 y $75.000: 651


,ID_Transaccion,Sucursal,Genero_Libro,Formato,Precio_Lista,Unidades_Vendidas,Descuento_Aplicado,Metodo_Pago
0,LIB-9001,Galería de Arte,Ensayo Científico,Bolsillo,72000.0,1,0.0,Efectivo
1,LIB-9002,Centro Histórico,Historia y Crítica,Edición Ilustrada,45000.0,1,0.1,Efectivo
3,LIB-9004,Galería de Arte,Ensayo Científico,Tapa Blanda,45000.0,5,0.0,Tarjeta Crédito
6,LIB-9007,Terminal,Ensayo Científico,Tapa Blanda,45000.0,3,NaN,Tarjeta Débito
7,LIB-9008,Zona Rosa,Ensayo Científico,Tapa Blanda,58000.0,3,0.0,Tarjeta Débito


## 8. Consulta directa con `.query()`
Ventas pagadas por `Transferencia QR` con al menos 2 unidades.


In [8]:
ventas_qr = df_libros.query(
    "Metodo_Pago == 'Transferencia QR' and Unidades_Vendidas >= 2"
)
print("Operaciones que cumplen la consulta:", len(ventas_qr))
display(ventas_qr.head())


Operaciones que cumplen la consulta: 183


,ID_Transaccion,Sucursal,Genero_Libro,Formato,Precio_Lista,Unidades_Vendidas,Descuento_Aplicado,Metodo_Pago
5,LIB-9006,Zona Rosa,Literatura Clásica,Edición Ilustrada,89000.0,4,0.0,Transferencia QR
9,LIB-9010,Zona Rosa,Historia y Crítica,Edición Ilustrada,32000.0,5,0.0,Transferencia QR
18,LIB-9019,Centro Histórico,Novela Gráfica,Edición Ilustrada,72000.0,2,0.0,Transferencia QR
20,LIB-9021,Centro Histórico,Novela Gráfica,Bolsillo,58000.0,6,0.0,Transferencia QR
22,LIB-9023,Centro Histórico,Ensayo Científico,Bolsillo,58000.0,2,0.0,Transferencia QR


## 9. Filtrado compuesto con métodos de texto
Se seleccionan libros cuyo género comienza por `Ensayo` y cuyo precio es superior a $50.000. `na=False` evita problemas si hubiera géneros ausentes.


In [9]:
condicion_ensayo_precio = (
    df_libros["Genero_Libro"].str.startswith("Ensayo", na=False)
    & (df_libros["Precio_Lista"] > 50000)
)
ventas_ensayo = df_libros[condicion_ensayo_precio]
print("Libros de género Ensayo con precio superior a $50.000:", len(ventas_ensayo))
display(ventas_ensayo.head())


Libros de género Ensayo con precio superior a $50.000: 84


,ID_Transaccion,Sucursal,Genero_Libro,Formato,Precio_Lista,Unidades_Vendidas,Descuento_Aplicado,Metodo_Pago
0,LIB-9001,Galería de Arte,Ensayo Científico,Bolsillo,72000.0,1,0.0,Efectivo
7,LIB-9008,Zona Rosa,Ensayo Científico,Tapa Blanda,58000.0,3,0.0,Tarjeta Débito
10,LIB-9011,Terminal,Ensayo Científico,Bolsillo,58000.0,4,0.0,Tarjeta Débito
22,LIB-9023,Centro Histórico,Ensayo Científico,Bolsillo,58000.0,2,0.0,Transferencia QR
49,LIB-9050,Campus Universitario,Ensayo Científico,Bolsillo,58000.0,2,0.1,Efectivo


## 10. Imputación de valores faltantes con `.fillna()`
Los precios faltantes se reemplazan por el promedio aritmético de `Precio_Lista`. Los descuentos faltantes se reemplazan por `0.0` (sin descuento).


In [10]:
nulos_precio_antes = df_libros["Precio_Lista"].isna().sum()
nulos_descuento_antes = df_libros["Descuento_Aplicado"].isna().sum()
promedio_precio = df_libros["Precio_Lista"].mean()

df_libros["Precio_Lista"] = df_libros["Precio_Lista"].fillna(promedio_precio)
df_libros["Descuento_Aplicado"] = df_libros["Descuento_Aplicado"].fillna(0.0)

print(f"Promedio utilizado para imputar Precio_Lista: ${promedio_precio:,.2f}")
print("Nulos de Precio_Lista antes:", nulos_precio_antes)
print("Nulos de Descuento_Aplicado antes:", nulos_descuento_antes)
print("Nulos restantes en Precio_Lista:", df_libros["Precio_Lista"].isna().sum())
print("Nulos restantes en Descuento_Aplicado:", df_libros["Descuento_Aplicado"].isna().sum())


Promedio utilizado para imputar Precio_Lista: $57,003.16
Nulos de Precio_Lista antes: 50
Nulos de Descuento_Aplicado antes: 88
Nulos restantes en Precio_Lista: 0
Nulos restantes en Descuento_Aplicado: 0


## 11. Transformación con función y `.apply()`
La función calcula el subtotal neto según la fórmula indicada: `(Precio_Lista * (1 - Descuento_Aplicado)) * Unidades_Vendidas`. Se asume que el descuento está expresado como proporción decimal (por ejemplo, `0.10` equivale al 10 %).


In [11]:
def calcular_total_venta(fila):
    return (
        fila["Precio_Lista"]
        * (1 - fila["Descuento_Aplicado"])
        * fila["Unidades_Vendidas"]
    )

df_libros["Total_Venta"] = df_libros.apply(calcular_total_venta, axis=1)
display(df_libros[[
    "ID_Transaccion", "Precio_Lista", "Descuento_Aplicado",
    "Unidades_Vendidas", "Total_Venta"
]].head())


,ID_Transaccion,Precio_Lista,Descuento_Aplicado,Unidades_Vendidas,Total_Venta
0,LIB-9001,72000.0,0.00,1,72000.0
1,LIB-9002,45000.0,0.10,1,40500.0
2,LIB-9003,115000.0,0.00,1,115000.0
3,LIB-9004,45000.0,0.00,5,225000.0
4,LIB-9005,32000.0,0.05,2,60800.0


## 12. Renombrado y reorganización de variables
Se renombran `Genero_Libro` a `Categoria` y `Descuento_Aplicado` a `Descuento`. Después, `ID_Transaccion` y `Total_Venta` quedan al inicio.


In [12]:
df_libros = df_libros.rename(columns={
    "Genero_Libro": "Categoria",
    "Descuento_Aplicado": "Descuento"
})

columnas_inicio = ["ID_Transaccion", "Total_Venta"]
otras_columnas = [col for col in df_libros.columns if col not in columnas_inicio]
df_libros = df_libros[columnas_inicio + otras_columnas]

print("Orden final de las columnas:")
print(df_libros.columns.tolist())
display(df_libros.head())


Orden final de las columnas:
['ID_Transaccion', 'Total_Venta', 'Sucursal', 'Categoria', 'Formato', 'Precio_Lista', 'Unidades_Vendidas', 'Descuento', 'Metodo_Pago']


,ID_Transaccion,Total_Venta,Sucursal,Categoria,Formato,Precio_Lista,Unidades_Vendidas,Descuento,Metodo_Pago
0,LIB-9001,72000.0,Galería de Arte,Ensayo Científico,Bolsillo,72000.0,1,0.00,Efectivo
1,LIB-9002,40500.0,Centro Histórico,Historia y Crítica,Edición Ilustrada,45000.0,1,0.10,Efectivo
2,LIB-9003,115000.0,Terminal,Novela Gráfica,Bolsillo,115000.0,1,0.00,Tarjeta Débito
3,LIB-9004,225000.0,Galería de Arte,Ensayo Científico,Tapa Blanda,45000.0,5,0.00,Tarjeta Crédito
4,LIB-9005,60800.0,Centro Histórico,Literatura Clásica,Tapa Blanda,32000.0,2,0.05,Tarjeta Crédito


## 13. Métricas de facturación por sucursal
Se agrupa por `Sucursal` y se calcula el promedio de `Precio_Lista` y la suma de `Total_Venta`.


In [13]:
resumen_sucursal = df_libros.groupby("Sucursal").agg(
    Precio_Lista_Promedio=("Precio_Lista", "mean"),
    Facturacion_Total=("Total_Venta", "sum")
).sort_values("Facturacion_Total", ascending=False)

display(resumen_sucursal)


,Precio_Lista_Promedio,Facturacion_Total
Sucursal,,
Galería de Arte,55927.444976,4.098825e+07
Campus Universitario,57629.974696,3.896883e+07
Centro Histórico,57419.013502,3.559460e+07
Terminal,58516.091825,3.365604e+07
Zona Rosa,55668.541042,3.341374e+07


## 14. Métricas de rango personalizadas por formato
La función `rango` calcula máximo menos mínimo. Se aplica a `Precio_Lista` y `Total_Venta` agrupando por `Formato`.


In [14]:
def rango(series):
    return series.max() - series.min()

resumen_rango_formato = df_libros.groupby("Formato").agg(
    Rango_Precio_Lista=("Precio_Lista", rango),
    Rango_Total_Venta=("Total_Venta", rango)
)

display(resumen_rango_formato)


,Rango_Precio_Lista,Rango_Total_Venta
Formato,,
Bolsillo,83000.0,503600.0
Edición Ilustrada,83000.0,544600.0
Tapa Blanda,83000.0,662800.0
Tapa Dura,83000.0,664400.0


## 15. Agrupamiento multinivel por sucursal y categoría
Para cada combinación de `Sucursal` y `Categoria`, se calcula el promedio de `Total_Venta` y la suma de `Unidades_Vendidas`.


In [15]:
resumen_sucursal_categoria = df_libros.groupby(
    ["Sucursal", "Categoria"]
).agg(
    Total_Venta_Promedio=("Total_Venta", "mean"),
    Unidades_Vendidas_Total=("Unidades_Vendidas", "sum")
).sort_index()

display(resumen_sucursal_categoria)


Total_Venta_Promedio  \
Sucursal             Categoria                                  
Campus Universitario Ciencia Ficción            151050.563910   
                     Ensayo Científico          185033.969474   
                     Historia y Crítica         213691.003509   
                     Literatura Clásica         195582.724458   
                     Novela Gráfica             192083.783784   
Centro Histórico     Ciencia Ficción            167983.421053   
                     Ensayo Científico          202911.695906   
                     Historia y Crítica         200169.170526   
                     Literatura Clásica         181721.866165   
                     Novela Gráfica             179066.401469   
Galería de Arte      Ciencia Ficción            163494.514874   
                     Ensayo Científico          158508.530702   
                     Historia y Crítica         210753.056530   
                     Literatura Clásica         209879.411765   
                     Novela Gráfica             193224.357341   
Terminal             Ciencia Ficción            196755.078947   
                     Ensayo Científico          182576.856140   
                     Historia y Crítica         187277.340909   
                     Literatura Clásica         167968.778393   
                     Novela Gráfica             157931.051170   
Zona Rosa            Ciencia Ficción            169218.430162   
                     Ensayo Científico          190466.666667   
                     Historia y Crítica         177175.426316   
                     Literatura Clásica         173039.713450   
                     Novela Gráfica             164066.623877   

                                         Unidades_Vendidas_Total  
Sucursal             Categoria                                    
Campus Universitario Ciencia Ficción                         121  
                     Ensayo Científico                       173  
                     Historia y Crítica                      174  
                     Literatura Clásica                      111  
                     Novela Gráfica                          134  
Centro Histórico     Ciencia Ficción                         118  
                     Ensayo Científico                       104  
                     Historia y Crítica                      181  
                     Literatura Clásica                      118  
                     Novela Gráfica                          138  
Galería de Arte      Ciencia Ficción                         148  
                     Ensayo Científico                       161  
                     Historia y Crítica                      193  
                     Literatura Clásica                      125  
                     Novela Gráfica                          140  
Terminal             Ciencia Ficción                         136  
                     Ensayo Científico                       106  
                     Historia y Crítica                      150  
                     Literatura Clásica                      112  
                     Novela Gráfica                          102  
Zona Rosa            Ciencia Ficción                         164  
                     Ensayo Científico                        92  
                     Historia y Crítica                      134  
                     Literatura Clásica                      124  
                     Novela Gráfica                          133

## 16. Comprobaciones finales
Se verifican las dimensiones de la tabla procesada y que las columnas imputadas no tengan valores ausentes.


In [16]:
print("Dimensiones finales:", df_libros.shape)
print("Duplicados completos restantes:", df_libros.duplicated().sum())
print("Nulos restantes en columnas numéricas principales:")
display(df_libros[["Precio_Lista", "Descuento", "Total_Venta"]].isna().sum().to_frame("Nulos"))


Dimensiones finales: (1000, 9)
Duplicados completos restantes: 0
Nulos restantes en columnas numéricas principales:


,Nulos
Precio_Lista,0
Descuento,0
Total_Venta,0
